# Unsloth QLoRA Smoke Test on AMD Radeon Pro W7900 (ROCm)

Fine-tune Qwen3-4B with 4-bit QLoRA on a single AMD RDNA3 GPU using Unsloth.

This notebook is a workshop smoke test: it proves the full ROCm + Unsloth + bitsandbytes
+ TRL training stack works end to end on AMD silicon, with a visible loss drop and a
measured peak VRAM footprint. Each cell verifies a real capability, not a version string.

Hardware: AMD Radeon Pro W7900 (Navi 31, gfx1100, 48 GB)
Stack: torch 2.10.0+rocm7.0, Unsloth Studio venv, bitsandbytes (ROCm)


## 1. Verify the ROCm torch stack sees the GPU

In [1]:
import torch, os
print("torch", torch.__version__, "| hip", torch.version.hip)
print("cuda(hip) available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
assert torch.cuda.is_available(), "GPU not visible -- check the render group + ROCm torch wheel"
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))


torch 2.10.0+rocm7.0 | hip 7.0.51831
cuda(hip) available: True
device count: 1
  0 AMD Radeon Graphics


/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


## 2. Prove real GPU compute (matmul TFLOP/s)
A clean matmul on the device confirms the HIP backend is live, not just importable.

In [2]:
import time
a = torch.randn(8192, 8192, device="cuda", dtype=torch.float16)
b = torch.randn(8192, 8192, device="cuda", dtype=torch.float16)
torch.cuda.synchronize(); t = time.time()
for _ in range(20):
    c = a @ b
torch.cuda.synchronize()
dt = (time.time() - t) / 20
tflops = 2 * 8192**3 / dt / 1e12
print(f"fp16 matmul: {dt*1e3:.2f} ms/iter   {tflops:.1f} TFLOP/s")
del a, b, c; torch.cuda.empty_cache()


/tmp/ipykernel_28386/1073903646.py:2: UserWarning: expandable_segments not supported on this platform (Triggered internally at /pytorch/c10/hip/HIPAllocatorConfig.h:40.)
  a = torch.randn(8192, 8192, device="cuda", dtype=torch.float16)


fp16 matmul: 16.03 ms/iter   68.6 TFLOP/s


## 3. Load Qwen3-4B in 4-bit with Unsloth
`fast_inference=False` -> no vLLM needed (Unsloth native path). 4-bit keeps VRAM small.

In [3]:
from unsloth import FastLanguageModel
max_seq_length = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    fast_inference = False,   # vLLM-only if True; native path here
)
print("model loaded:", type(model).__name__)


[bitsandbytes.cuda_specs|ERROR]Could not detect ROCm GPU architecture: [Errno 2] No such file or directory: 'rocminfo'


[bitsandbytes.cuda_specs|WARNING]
ROCm GPU architecture detection failed despite ROCm being available.
                


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/shailen/.unsloth/studio/unsloth_studio/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.7.2: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    AMD gfx1100 GPU. Num GPUs = 1. Max memory: 44.984 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+rocm7.0. ROCm Toolkit: 7.0.51831. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model loaded: Qwen3ForCausalLM


## 4. Attach LoRA adapters

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.3f}%)")


Unsloth 2026.7.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 33,030,144 / 2,541,616,640  (1.300%)


## 5. Prepare a tiny slice of FineTome for the smoke test

In [5]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt, get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
n = 1000
dataset = load_dataset("mlabonne/FineTome-100k", split=f"train[:{n}]")
dataset = standardize_sharegpt(dataset)

def fmt(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
             for c in convos]
    return {"text": texts}

dataset = dataset.map(fmt, batched=True)
print("examples:", len(dataset))
print(dataset[0]["text"][:400])


Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating train split:  17%|█▋        | 17000/100000 [00:00<00:00, 161861.73 examples/s]

Generating train split:  46%|████▌     | 46000/100000 [00:00<00:00, 229979.45 examples/s]

Generating train split:  76%|███████▌  | 76000/100000 [00:00<00:00, 258001.93 examples/s]

Generating train split: 100%|██████████| 100000/100000 [00:00<00:00, 252672.26 examples/s]

...

Unsloth: Standardizing formats (num_proc=25):   4%|▍         | 40/1000 [00:00<00:20, 46.78 examples/s]

Unsloth: Standardizing formats (num_proc=25): 100%|██████████| 1000/1000 [00:01<00:00, 963.89 examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map: 100%|██████████| 1000/1000 [00:00<00:00, 22004.18 examples/s]

examples: 1000
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-c


## 6. Train (QLoRA via TRL SFTTrainer)
Effective batch 16 (1 x 16 grad-accum). SMOKE=1 -> 30 steps; else 60.

In [6]:
import os
from trl import SFTTrainer, SFTConfig

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
max_steps = 30 if os.environ.get("SMOKE") == "1" else 60

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 16,
        warmup_steps = 5,
        max_steps = max_steps,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

torch.cuda.reset_peak_memory_stats()
stats = trainer.train()
peak_gb = torch.cuda.max_memory_reserved() / 1e9
print(f"\ntrain runtime: {stats.metrics['train_runtime']:.1f}s")
print(f"peak reserved VRAM: {peak_gb:.2f} GB")


Unsloth: Tokenizing ["text"] (num_proc=25):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=25):   4%|▍         | 40/1000 [00:01<00:29, 32.53 examples/s]

Unsloth: Tokenizing ["text"] (num_proc=25):  16%|█▌        | 160/1000 [00:01<00:05, 147.87 examples/s]

Unsloth: Tokenizing ["text"] (num_proc=25):  24%|██▍       | 240/1000 [00:01<00:03, 218.26 examples/s]

Unsloth: Tokenizing ["text"] (num_proc=25):  32%|███▏      | 320/1000 [00:01<00:02, 284.26 examples/s]

...

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.843800
2,1.673300
3,1.462700
4,1.585300
5,1.613300
6,1.472300
7,1.224500
8,1.150600
9,1.157500
10,1.139000



train runtime: 297.0s
peak reserved VRAM: 5.56 GB


## 7. Confirm the loss actually dropped
A smoke test passes only if the model learned: first-step loss > last-step loss.

In [7]:
hist = [h for h in trainer.state.log_history if "loss" in h]
first, last = hist[0]["loss"], hist[-1]["loss"]
print(f"loss: {first:.3f} -> {last:.3f}  (delta {first-last:+.3f})")
assert last < first, "Loss did not decrease -- training smoke test FAILED"
print("SMOKE TEST PASSED: training reduced the loss on AMD ROCm.")


loss: 1.844 -> 0.813  (delta +1.031)
SMOKE TEST PASSED: training reduced the loss on AMD ROCm.


## 8. Quick inference sanity check

In [8]:
FastLanguageModel.for_inference(model)
msgs = [{"role": "user", "content": "In one sentence, what is QLoRA?"}]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True,
            add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=80, temperature=0.7, top_p=0.9)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In one sentence, QLoRA is a technique that enables the training of large language models on devices with limited resources.


## 9. Save the LoRA adapter

In [9]:
save_dir = "qwen3_4b_qlora_smoketest_adapter"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
import os
print("saved:", sorted(os.listdir(save_dir)))


saved: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'chat_template.jinja', 'merges.txt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']
